# South Sudan Health & Humanitarian Dashboard - JupyterLite version

This notebook is a JupyterLite/Pyodide conversion of the original Shiny app. It uses `ipywidgets` for controls, `folium` for the map, `plotly` for charts, and a pre-rendered DEM PNG instead of `rasterio`.

Run the cells from top to bottom. In JupyterLite, the first install cell may take a moment the first time it runs.

In [ ]:
# JupyterLite/Pyodide setup
try:
    import piplite
    await piplite.install(['pandas', 'numpy', 'plotly', 'folium', 'ipywidgets'])
except Exception as exc:
    print('Package install step skipped or already satisfied:', exc)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import MarkerCluster
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

DATA_DIR = Path('data')
FACILITIES_CSV = DATA_DIR / 'SouthSudanHealthFacilities1282026.csv'
HSTP_JSON = DATA_DIR / 'hstp_discontinued.json'
DEM_PNG = DATA_DIR / 'dem_overlay.png'
DEM_META = DATA_DIR / 'dem_bounds.json'
H3_RES5 = DATA_DIR / 'SouthSudanH3res5.geojson'
H3_RES6 = DATA_DIR / 'SouthSudanH3res6.geojson'


In [ ]:
def load_facilities():
    df = pd.read_csv(FACILITIES_CSV)
    df = df[(df['Latitude'] != 0) & (df['Longitude'] != 0)]
    df = df[df['Latitude'].notna() & df['Longitude'].notna()].copy()

    with open(HSTP_JSON, 'r') as f:
        hstp = set(json.load(f))

    df['Functional'] = df['Functional'].astype(str)
    df['IsFunctional'] = df['Functional'].str.lower().str.contains('functional') & ~df['Functional'].str.lower().str.contains('non')
    df['Facility_T'] = df['Facility_T'].astype(str)
    df['Facility_N'] = df['Facility_N'].astype(str)
    df['IsHospital'] = df['Facility_T'].str.contains('Hospital', na=False) | (df['Facility_T'] == 'TH')
    df['IsHSTP'] = df['Facility_N'].apply(lambda x: str(x).replace('/', ' ').strip() in hstp)
    for col in ['State', 'County', 'Facility_N', 'Facility_T']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace('/', ' ', regex=False).str.strip()
    return df


def create_sample_acled_data(n=80, seed=42):
    rng = np.random.default_rng(seed)
    states = ['Central Equatoria', 'Upper Nile', 'Jonglei', 'Unity', 'Western Equatoria', 'Warrap', 'Lakes']
    dates = pd.date_range(end=pd.Timestamp.today(), periods=n, freq='3D')
    return pd.DataFrame({
        'event_date': dates,
        'latitude': rng.uniform(4.0, 10.0, n),
        'longitude': rng.uniform(27.0, 34.0, n),
        'event_type': rng.choice(['Battles', 'Violence against civilians', 'Riots', 'Explosions/Remote violence'], n),
        'fatalities': rng.choice([0, 1, 2, 3, 5, 8, 12], n),
        'admin1': rng.choice(states, n),
    })


def create_sample_idp_data(n=50, seed=7):
    rng = np.random.default_rng(seed)
    states = ['Central Equatoria', 'Upper Nile', 'Jonglei', 'Unity', 'Western Equatoria', 'Warrap', 'Lakes']
    return pd.DataFrame({
        'event_start_date': pd.date_range(end=pd.Timestamp.today(), periods=n, freq='7D'),
        'admin1': rng.choice(states, n),
        'admin2': rng.choice(['Sample County A', 'Sample County B', 'Sample County C'], n),
        'figure': rng.integers(100, 5000, n),
        'displacement_type': rng.choice(['Conflict', 'Disaster', 'Other'], n),
        'latitude': rng.uniform(4.0, 10.0, n),
        'longitude': rng.uniform(27.0, 34.0, n),
    })


def try_load_hdx_idp():
    url = 'https://data.humdata.org/dataset/b22850c5-cece-4382-9dbf-0b30b4090ee7/resource/78eb1954-8150-4e6b-bafd-c0b39285e637/download/event_data_ssd.csv'
    try:
        df = pd.read_csv(url)
        if 'event_start_date' in df.columns:
            df['event_start_date'] = pd.to_datetime(df['event_start_date'], errors='coerce')
        if 'figure' in df.columns:
            df['figure'] = pd.to_numeric(df['figure'], errors='coerce').fillna(0)
        return df
    except Exception as exc:
        print('HDX live load unavailable in this JupyterLite session; using sample displacement data.', exc)
        return create_sample_idp_data()


df_facilities = load_facilities()
df_acled = create_sample_acled_data()  # Replace with an exported ACLED CSV when available.
df_idp = try_load_hdx_idp()

print(f'Facilities: {len(df_facilities):,}')
print(f'Conflict events: {len(df_acled):,} (sample/offline)')
print(f'Displacement rows: {len(df_idp):,}')


In [ ]:
def filter_data(state='All States', status='All', period='Last 180 days', search=''):
    fac = df_facilities.copy()
    if state != 'All States':
        fac = fac[fac['State'] == state]
    if status == 'Functional':
        fac = fac[fac['IsFunctional']]
    elif status == 'Non-Functional':
        fac = fac[~fac['IsFunctional']]
    elif status == 'HSTP Only':
        fac = fac[fac['IsHSTP']]
    if search:
        s = search.lower()
        fac = fac[
            fac['Facility_N'].str.lower().str.contains(s, na=False) |
            fac['County'].str.lower().str.contains(s, na=False) |
            fac['State'].str.lower().str.contains(s, na=False)
        ]

    days = {'Last 30 days': 30, 'Last 90 days': 90, 'Last 180 days': 180, 'Last Year': 365}.get(period, 180)
    cutoff = pd.Timestamp.today() - pd.Timedelta(days=days)

    conf = df_acled.copy()
    if 'event_date' in conf.columns:
        conf = conf[pd.to_datetime(conf['event_date'], errors='coerce') >= cutoff]
    if state != 'All States' and 'admin1' in conf.columns:
        conf = conf[conf['admin1'] == state]

    idp = df_idp.copy()
    if 'event_start_date' in idp.columns:
        idp = idp[pd.to_datetime(idp['event_start_date'], errors='coerce') >= cutoff]
    if state != 'All States' and 'admin1' in idp.columns:
        idp = idp[idp['admin1'] == state]
    return fac, conf, idp


def metric_card(value, label, background):
    return f"""
    <div style='background:{background}; color:white; padding:16px; border-radius:10px; min-width:150px; margin:5px; box-shadow:0 2px 6px rgba(0,0,0,.12);'>
      <div style='font-size:28px; font-weight:700; line-height:1;'>{value}</div>
      <div style='font-size:12px; opacity:.92; margin-top:6px;'>{label}</div>
    </div>
    """


def render_metrics(fac, conf, idp):
    functional = int(fac['IsFunctional'].sum()) if len(fac) else 0
    rate = functional / len(fac) * 100 if len(fac) else 0
    fatalities = float(conf['fatalities'].sum()) if 'fatalities' in conf.columns else 0
    idps = float(idp['figure'].sum()) if 'figure' in idp.columns else 0
    html = ''.join([
        metric_card(f'{len(fac):,}', 'Health Facilities', 'linear-gradient(135deg,#3b82f6,#1e40af)'),
        metric_card(f'{functional:,}', f'Functional ({rate:.1f}%)', 'linear-gradient(135deg,#22c55e,#15803d)'),
        metric_card(f'{int(fac["IsHSTP"].sum()):,}', 'HSTP Discontinued', 'linear-gradient(135deg,#f97316,#c2410c)'),
        metric_card(f'{len(conf):,}', 'Conflict Events', 'linear-gradient(135deg,#7c2d12,#991b1b)'),
        metric_card(f'{fatalities:.0f}', 'Fatalities', 'linear-gradient(135deg,#450a0a,#7f1d1d)'),
        metric_card(f'{idps:,.0f}', 'IDPs Displaced', 'linear-gradient(135deg,#7c3aed,#5b21b6)'),
    ])
    display(HTML(f"<div style='display:flex; flex-wrap:wrap'>{html}</div>"))


def make_map(fac, conf, idp, layers):
    m = folium.Map(location=[7.3, 30.2], zoom_start=6, tiles='OpenStreetMap')
    lats, lons = [], []

    if 'DEM' in layers and DEM_PNG.exists() and DEM_META.exists():
        meta = json.loads(DEM_META.read_text())
        folium.raster_layers.ImageOverlay(name='Elevation DEM overlay', image=str(DEM_PNG), bounds=meta['bounds'], opacity=0.65, interactive=False, zindex=1).add_to(m)

    if 'H3 res5' in layers and H3_RES5.exists():
        folium.GeoJson(str(H3_RES5), name='H3 res5', style_function=lambda _: {'fillOpacity': 0.02, 'weight': 0.5}).add_to(m)
    if 'H3 res6' in layers and H3_RES6.exists():
        folium.GeoJson(str(H3_RES6), name='H3 res6', style_function=lambda _: {'fillOpacity': 0.02, 'weight': 0.3}).add_to(m)

    if 'Facilities' in layers:
        cluster = MarkerCluster(name='Health Facilities').add_to(m)
        for _, row in fac.iterrows():
            color = 'orange' if row['IsHSTP'] else ('blue' if row['IsHospital'] and row['IsFunctional'] else ('green' if row['IsFunctional'] else 'red'))
            icon = 'exclamation-triangle' if row['IsHSTP'] else ('hospital' if row['IsHospital'] else 'medkit')
            popup = f"<b>{row['Facility_N']}</b><br>Type: {row['Facility_T']}<br>County: {row['County']}<br>Status: {row['Functional']}"
            folium.Marker([row['Latitude'], row['Longitude']], popup=popup, tooltip=row['Facility_N'], icon=folium.Icon(color=color, icon=icon, prefix='fa')).add_to(cluster)
        lats += fac['Latitude'].dropna().tolist(); lons += fac['Longitude'].dropna().tolist()

    if 'Conflict' in layers and {'latitude', 'longitude'}.issubset(conf.columns):
        group = folium.FeatureGroup(name='Conflict Events').add_to(m)
        for _, row in conf.iterrows():
            if pd.isna(row.get('latitude')) or pd.isna(row.get('longitude')):
                continue
            radius = 5 + float(row.get('fatalities', 0)) * 1.5
            folium.CircleMarker([row['latitude'], row['longitude']], radius=radius, popup=f"{row.get('event_type','Unknown')}<br>{int(row.get('fatalities',0))} fatalities", color='#7c2d12', fillColor='#991b1b', fillOpacity=0.6).add_to(group)
        lats += conf['latitude'].dropna().tolist(); lons += conf['longitude'].dropna().tolist()

    if 'IDP' in layers and {'latitude', 'longitude'}.issubset(idp.columns):
        group = folium.FeatureGroup(name='IDP Events').add_to(m)
        for _, row in idp.iterrows():
            if pd.isna(row.get('latitude')) or pd.isna(row.get('longitude')):
                continue
            figure = float(row.get('figure', 0) or 0)
            folium.CircleMarker([row['latitude'], row['longitude']], radius=max(5, min(figure / 250, 22)), popup=f"{int(figure):,} displaced<br>{row.get('displacement_type','Unknown')}", color='#7c3aed', fillColor='#a78bfa', fillOpacity=0.5).add_to(group)
        lats += idp['latitude'].dropna().tolist(); lons += idp['longitude'].dropna().tolist()

    if lats and lons:
        m.fit_bounds([[min(lats)-0.25, min(lons)-0.25], [max(lats)+0.25, max(lons)+0.25]], padding=(20, 20))
    folium.LayerControl().add_to(m)
    return m


In [ ]:
def show_health_charts(fac):
    if len(fac) == 0:
        display(HTML('<b>No facilities match the current filters.</b>'))
        return
    stats = fac.groupby('State').agg(Total=('Facility_N','count'), Functional=('IsFunctional','sum')).reset_index()
    stats['Non-Functional'] = stats['Total'] - stats['Functional']
    stats = stats.sort_values('Total', ascending=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(y=stats['State'], x=stats['Functional'], name='Functional', orientation='h'))
    fig.add_trace(go.Bar(y=stats['State'], x=stats['Non-Functional'], name='Non-Functional', orientation='h'))
    fig.update_layout(title='Health Facilities by State', barmode='stack', height=500, template='plotly_white')
    fig.show()
    counts = fac['Facility_T'].value_counts().reset_index()
    counts.columns = ['Type','Count']
    px.pie(counts, values='Count', names='Type', title='Facility Type Distribution', hole=0.4).show()


def show_conflict_charts(conf):
    if len(conf) == 0:
        display(HTML('<b>No conflict events match the current filters.</b>'))
        return
    daily = conf.groupby(pd.to_datetime(conf['event_date']).dt.date).agg(Events=('fatalities','size'), Fatalities=('fatalities','sum')).reset_index().rename(columns={'event_date':'Date'})
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=daily['Date'], y=daily['Events'], name='Events'))
    fig.add_trace(go.Bar(x=daily['Date'], y=daily['Fatalities'], name='Fatalities', opacity=0.65))
    fig.update_layout(title='Conflict Events Over Time', height=400, template='plotly_white')
    fig.show()
    if 'event_type' in conf.columns:
        types = conf.groupby('event_type').agg(Events=('event_type','size'), Fatalities=('fatalities','sum')).reset_index()
        px.bar(types, x='event_type', y=['Events','Fatalities'], title='Conflict by Type', barmode='group').show()


def show_idp_charts(idp):
    if len(idp) == 0 or 'figure' not in idp.columns:
        display(HTML('<b>No displacement records match the current filters.</b>'))
        return
    if 'event_start_date' in idp.columns:
        monthly = idp.groupby(pd.to_datetime(idp['event_start_date']).dt.to_period('M')).agg(figure=('figure','sum')).reset_index()
        monthly['event_start_date'] = monthly['event_start_date'].astype(str)
        px.area(monthly, x='event_start_date', y='figure', title='Displacement Over Time', labels={'figure':'IDPs','event_start_date':'Month'}).show()
    if 'displacement_type' in idp.columns:
        cause = idp.groupby('displacement_type').agg(IDPs=('figure','sum')).reset_index()
        px.bar(cause, x='displacement_type', y='IDPs', title='Displacement by Cause').show()


def show_tables(fac, conf, idp):
    display(HTML('<h3>State summary</h3>'))
    summary = fac.groupby('State').agg(Total=('Facility_N','count'), Functional=('IsFunctional','sum'), HSTP=('IsHSTP','sum')).reset_index()
    summary['Non-Functional'] = summary['Total'] - summary['Functional']
    summary['Functional %'] = (summary['Functional'] / summary['Total'] * 100).round(1)
    display(summary.sort_values('HSTP', ascending=False))
    display(HTML('<h3>Facilities</h3>'))
    display(fac[['State','County','Facility_N','Facility_T','Functional','IsHSTP']].head(200))
    display(HTML('<h3>Conflict events</h3>'))
    cols = [c for c in ['event_date','event_type','fatalities','admin1','latitude','longitude'] if c in conf.columns]
    display(conf[cols].sort_values('event_date', ascending=False).head(100) if cols else conf.head(100))
    display(HTML('<h3>IDP events</h3>'))
    cols = [c for c in ['event_start_date','figure','admin1','admin2','displacement_type','latitude','longitude'] if c in idp.columns]
    display(idp[cols].sort_values(cols[0], ascending=False).head(100) if cols else idp.head(100))


In [ ]:
states = ['All States'] + sorted(df_facilities['State'].dropna().unique().tolist())
state_w = widgets.Dropdown(options=states, value='All States', description='State:')
status_w = widgets.Dropdown(options=['All', 'Functional', 'Non-Functional', 'HSTP Only'], value='All', description='Status:')
period_w = widgets.Dropdown(options=['Last 30 days', 'Last 90 days', 'Last 180 days', 'Last Year'], value='Last 180 days', description='Period:')
search_w = widgets.Text(value='', placeholder='Facility, county, state...', description='Search:')
layers_w = widgets.SelectMultiple(options=['Facilities', 'Conflict', 'IDP', 'DEM', 'H3 res5', 'H3 res6'], value=('Facilities', 'Conflict', 'IDP'), description='Layers:', rows=6)
view_w = widgets.ToggleButtons(options=['Map', 'Health', 'Conflict', 'Displacement', 'Tables'], value='Map', description='View:')
refresh = widgets.Button(description='Refresh dashboard', button_style='primary')
out = widgets.Output()


def update_dashboard(*_):
    with out:
        clear_output(wait=True)
        fac, conf, idp = filter_data(state_w.value, status_w.value, period_w.value, search_w.value)
        display(HTML(f"""
        <div style='background:linear-gradient(90deg,#dc2626,#ea580c); color:white; padding:15px; border-radius:8px; margin-bottom:12px;'>
          <h2 style='margin:0 0 4px 0;'>South Sudan Health & Humanitarian Dashboard</h2>
          <div>Facilities: {len(fac):,} | Conflict events: {len(conf):,} | IDP rows: {len(idp):,}</div>
        </div>
        """))
        render_metrics(fac, conf, idp)
        if view_w.value == 'Map':
            display(make_map(fac, conf, idp, set(layers_w.value)))
        elif view_w.value == 'Health':
            show_health_charts(fac)
        elif view_w.value == 'Conflict':
            show_conflict_charts(conf)
        elif view_w.value == 'Displacement':
            show_idp_charts(idp)
        elif view_w.value == 'Tables':
            show_tables(fac, conf, idp)

refresh.on_click(update_dashboard)
for w in [state_w, status_w, period_w, search_w, layers_w, view_w]:
    w.observe(update_dashboard, names='value')

controls = widgets.VBox([widgets.HBox([state_w, status_w, period_w]), widgets.HBox([search_w, view_w, refresh]), layers_w])
display(controls, out)
update_dashboard()


In [ ]:
# Optional: export the current filtered facility table to CSV inside the JupyterLite file browser.
fac, conf, idp = filter_data(state_w.value, status_w.value, period_w.value, search_w.value)
outfile = Path('south_sudan_filtered_facilities.csv')
fac.to_csv(outfile, index=False)
print(f'Wrote {outfile} with {len(fac):,} rows')


## Conversion notes

- Shiny `input_*` controls were replaced with `ipywidgets`.
- Shiny outputs were converted to notebook display functions.
- The GeoTIFF DEM was pre-rendered to `data/dem_overlay.png` with `data/dem_bounds.json`; this avoids `rasterio`, which is not generally available in JupyterLite/Pyodide.
- The ACLED Python client was replaced with deterministic sample/offline data. For production use in JupyterLite, export ACLED events to CSV and load them like the facilities CSV.
- The HDX/IDMC loader attempts a browser-side CSV fetch; if CORS/network access fails, it falls back to sample data.
- The `Download All` Shiny handler is replaced by the CSV export cell.